# 04 · Evaluación por sujeto, ablación y eje de envejecimiento
[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Valoo-1201/vfc-caos-som/blob/main/notebooks/04_evaluacion_validacion.ipynb)

**Responsable:** Integrante C (con B) · **Entrada:** `results/features.csv` · **Salida:** `results/ablacion.csv`

**Regla de oro:** las ventanas de un mismo sujeto nunca pueden estar a la vez en entrenamiento y prueba. Por eso usamos `GroupKFold` y *leave-one-subject-out* agrupando por sujeto, y clasificamos a cada sujeto promediando el puntaje de sus ventanas.

In [ ]:
#@title 0 · Configuración (ejecutar siempre primero)
REPO = "Valoo-1201/vfc-caos-som"   # <- usuario/repositorio en GitHub
RAMA = "main"
USAR_DRIVE = True                  # guarda las series R-R en Google Drive para no descargarlas otra vez
MODO_DEMO = False                  # True = datos sintéticos (solo para probar el código sin internet)

import os, sys, subprocess
EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    os.chdir("/content")
    token = None
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")      # Colab -> 🔑 Secretos (solo hace falta si el repo es privado)
    except Exception:
        pass
    url = f"https://{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"
    if not os.path.exists("/content/vfc-caos-som"):
        subprocess.run(["git", "clone", "-q", "-b", RAMA, url, "/content/vfc-caos-som"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/vfc-caos-som", "pull", "-q"], check=False)
    os.chdir("/content/vfc-caos-som")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if USAR_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = "/content/drive/MyDrive/vfc-caos-som-data"
    else:
        DATA_DIR = "/content/data"
else:  # ejecución local: el notebook está en notebooks/
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
    DATA_DIR = "data"

sys.path[:0] = [os.path.abspath("src"), os.getcwd()]
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs("results/figures", exist_ok=True)

import numpy as np, pandas as pd, matplotlib.pyplot as plt
SEED = 42
np.random.seed(SEED)
pd.set_option("display.precision", 3)
print("Repositorio:", os.getcwd(), "| Datos:", DATA_DIR, "| Demo:", MODO_DEMO)

In [ ]:
from hrvchaos import metrics
from hrvchaos.evaluate import ablation
features = pd.read_csv("results/features.csv")
principal = features[features.group.isin(["RSN", "ICC"])]
print(principal.groupby("group").subject.nunique())

## Ablación: ¿aportan las métricas de caos?
Se compara el SOM (puntaje = proporción de ICC en la neurona ganadora y su vecindad) con una regresión logística sobre las mismas variables.

In [ ]:
conjuntos = {
    "lineales (SDNN, RMSSD…)": metrics.LINEAR,
    "no lineales": metrics.NONLINEAR,
    "entropías + Lyapunov": ["sampen", "mse_short", "mse_long", "lyap"],
    "todas": metrics.LINEAR + metrics.NONLINEAR,
}
abl = ablation(principal, conjuntos, positive="ICC", cv="groupkfold", n_splits=5, n_iter=5000, seed=SEED)
abl.to_csv("results/ablacion.csv", index=False)
abl

In [ ]:
#@title Validación leave-one-subject-out (más lenta, más estable con pocos sujetos)
loso = ablation(principal, {"todas": metrics.LINEAR + metrics.NONLINEAR}, positive="ICC", cv="loso",
                n_iter=3000, seed=SEED)
loso

## Intervalo de confianza por bootstrap
Con ~80 sujetos, una exactitud puntual engaña: reportar siempre el intervalo.

In [ ]:
from hrvchaos.evaluate import evaluate_som
rng = np.random.default_rng(SEED)
sujetos = principal.subject.unique()
boot = []
for b in range(20):   # subir a 200 para el informe final
    elegidos = rng.choice(sujetos, size=len(sujetos), replace=True)
    # las copias de un sujeto conservan su id, así siempre caen en el mismo pliegue (sin fuga)
    muestra = pd.concat([principal[principal.subject == s] for s in elegidos])
    boot.append(evaluate_som(muestra, metrics.LINEAR + metrics.NONLINEAR, n_iter=3000, seed=b)["SOM_bal_acc"])
print(f"Exactitud balanceada SOM: {np.mean(boot):.3f}  IC95% [{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]")

## Eje de envejecimiento (Fantasia)
Se entrena el SOM con RSN vs. ICC y se **proyectan** jóvenes y mayores sanos sin reentrenar. Si H4 es cierta, los mayores caen más cerca de la región ICC que los jóvenes.

In [ ]:
from sklearn.preprocessing import StandardScaler
from hrvchaos.som import SOM
FEATS = metrics.LINEAR + metrics.NONLINEAR
edad = features[features.group.isin(["Joven", "Mayor"])].dropna(subset=FEATS)
if len(edad) == 0:
    print("No hay datos de Fantasia en features.csv (añadir 'fantasia' en el notebook 02).")
else:
    p = principal.dropna(subset=FEATS)
    sc = StandardScaler().fit(p[FEATS])
    Xp = sc.transform(p[FEATS])
    som = SOM(*SOM.suggest_shape(len(Xp), Xp), len(FEATS), seed=SEED).fit(Xp, n_iter=20000)
    som.node_scores(Xp, p.group.values, "ICC")
    edad = edad.assign(puntaje_icc=som.predict_score(sc.transform(edad[FEATS])))
    por_sujeto = edad.groupby(["subject", "group"]).puntaje_icc.mean().reset_index()
    print(por_sujeto.groupby("group").puntaje_icc.describe())
    por_sujeto.boxplot(column="puntaje_icc", by="group"); plt.suptitle("")
    plt.title("Puntaje 'tipo ICC' en sanos jóvenes vs. mayores"); plt.show()

## Costo computacional (criterio de sostenibilidad)

In [ ]:
print(f"Tiempo medio por ventana de 2000 latidos: {features.seconds_to_compute.mean():.2f} s")
print(f"Tamaño del modelo SOM: {som.W.nbytes / 1024:.1f} KB" if 'som' in dir() else "")